In [ ]:
# imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import sklearn
from sklearn.preprocessing import MinMaxScaler

from scipy import stats

# Data preprocessing
### Includes any change made to the data before applying feature selection: imputation, normalisation, bootstrapping,

Process for CW1 went as follows:

Classification
1. One hot encoding (categorical features to binary) | Imputation (using mode, old method) 
Note: there were null values '?' that were not handled before one hot encoding
2. Normalisation (MinMax, numerical features -> 0-1 range)

Regression
1. Imputation (removed null data, mean for continuous, mode for categorical)
2. Normalisation (MinMax)
3. One hot encoding (pd.get_dummies)

Possible other methods for preprocessing include:
- Removing outliers
- VAE/Bootstrapping
    - Generating new instances from not a lot of data
        - Training data after removing outliers

And some more implicit methods include:
- Separating data into features and outcomes/labels

In [ ]:
# viewing data and datatypes (numerical / categorical)
dataset = pd.read_csv("./datasets/TrainDataset2025.csv")

dataset.info()

## Type-specific preprocessing
### Dataset datatypes include: object, float64 and int64
    # object: ID Column
    # float64: Continuous features
    # int64: Categorical features

### Handling object datatype
1. Remove object datatype

This only contains the ID column

In [ ]:
# Handling object datatype
print(dataset.select_dtypes(object)) # ID Column

print("Before: "+str(dataset.select_dtypes(object).shape))

dataset = dataset.drop("ID",axis=1)
print("After: "+str(dataset.select_dtypes(object).shape))


### Handling int64 datatype
#### Categorical Features
1. Remove non-categorical features

'original_shape_VoxelVolume' feature is int, but not categorical data (doesn't use classes)


In [ ]:
print(dataset.select_dtypes(np.int64))

print("Before: "+str(dataset.select_dtypes(np.int64).shape))

dataset['original_shape_VoxelVolume'] = dataset['original_shape_VoxelVolume'].astype(np.float64)
print("After: "+str(dataset.select_dtypes(np.int64).shape))

# Save column names of categorical features for later
categorical_names = dataset.select_dtypes(np.int64).columns
categorical_names

### Handling float64 datatype
#### Continuous Features

Nothing specific left, data is all continuous

In [ ]:
# Save column names of continuous features for later
dataset.select_dtypes(np.float64)
continuous_names = dataset.select_dtypes(np.float64).columns
continuous_names

## General Preprocessing
1. Null handling (converting '999')
2. Outlier handling
3. Data Imputation
4. Balancing data (Frequency handling)
5. Data Normalisation

### 1. Null Handling
Issue: Can't replace all '999' with np.nan. This converts int64 to float64 as NaN is float64
- Solution: by saving column names, in datatype-specific handling, for int64 (categorical) features, no longer a need to preserve int64 as no data is lost when converting to float

In [ ]:
# Replace 999 (placeholder) with actual null values
dataset = dataset.replace([999],np.nan)

# Viewing features with null values
non_null = dataset.isnull().sum()
non_null = non_null[non_null > 0]
non_null

### 2. Outlier Removal
Options: 
1. Z-Score
2. Interquartile Range

[Other Model-based methods:](https://www.geeksforgeeks.org/machine-learning/machine-learning-outlier/)

3. Isolation Forest
4. Local Outlier Factor
5. DBSCAN(?)

In [ ]:
# Data visualisation
scaler = MinMaxScaler()

# Boxplot below shows normalised continuous features + outliers
fig,ax=plt.subplots(1,figsize=(20,8))
sns.boxplot(data=scaler.fit_transform(dataset[continuous_names]),ax=ax)

Note: Above boxplot, categorical features normalised to 0-1 range, shows prevalence of outliers. Handling using naive methods, such as Z-score or IQR, may be ineffective

##### Option 1 - Z-score

In [ ]:
# Printing outlier locations
z_scores = np.abs(stats.zscore(dataset))
outliers_zscore = np.where(z_scores > 3)
outlier_indices = list(zip(outliers_zscore[0], outliers_zscore[1]))

print("Outlier total: "+str(len(outlier_indices)))
for index in outlier_indices:
    print(index)

In [ ]:
# Applying z-score

dataset[(np.abs(stats.zscore(dataset)) < 3).all(axis=1)] # < 3 checks for rows where all features are 3 standard deviations from the mean

Note: Z-score is clearly ineffective. Implication from above cell is that outliers are prevalent in all samples, hence they are all removed

##### Option 2 - Interquartile Range

In [ ]:
# Calculating Interquartile Range
Q1 = dataset.quantile(0.25)
Q3 = dataset.quantile(0.75)
IQR = Q3 - Q1

In [ ]:
# Outputting number of outliers per column
outliers_iqr = ((dataset < (Q1 - 1.5 * IQR)) | (dataset > (Q3 + 1.5 * IQR)))

print("Number of outliers per column:")
outliers_col = outliers_iqr.sum()
outliers_col = outliers_col[outliers_col > 0]

print(outliers_col)

In [ ]:
# Applying IQR
lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

upper_arr = np.where(dataset >= upper)[0]
lower_arr = np.where(dataset <= lower)[0]

# dataset.drop(index=upper_arr, inplace=True)
# dataset.drop(index=lower_arr, inplace=True)

### 3. Data Imputation
Performed after outlier handling as imputation should perform better with clean(er) data, especially for null values

Null values can be removed or imputed, or a combination of both consider only removing Gene null values, as other null totals are [minimal](#1-null-handling)  
- SimpleImputer for imputation by mean/mode/median
- IterativeImputer for model-based imputation

##### SimpleImputer

This has a 'missing_values' field, meaning we could set it to 999 instead of the earlier conversion to NaN
- Prefer not to do so as '999' would negatively affect algorithms

In [ ]:
from sklearn.impute import SimpleImputer

# Removal (entirely)
dataset = dataset.dropna()


# Removal (partial) + Imputation 
    # 1. Removing Gene
    # 2. Imputing other features (format: x|y -> continuous|categorical)
        # 2.1 Mean|Mode
        # 2.2 Mode|Mode

## 1.
dataset['Gene'].dropna()
## 2.


# Imputation (format: x|y -> continuous|categorical)
    
## Mode|Mean
imputer_cat = SimpleImputer(strategy='mean')
imputer_cont = SimpleImputer(strategy='most_frequent')

## Mode|Mode



dataset.info()

##### IterativeImputer

Allows for models to determine imputation

[This](https://scikit-learn.org/stable/modules/impute.html#multiple-vs-single-imputation:~:text=of%20IterativeImputer.-,7.4.3.2.%20Multiple%20vs.%20Single%20Imputation,-%23) might be helpful

### 4. Balancing
We can use stratification to make sure the data is split in the same way as the distribution
- Applies for classification as we split based on discrete outcome
- Question: How do we apply to regression?

Use a histogram to view relative frequencies

https://www.geeksforgeeks.org/machine-learning/data-preprocessing-machine-learning-python/#:~:text=Step%206%3A%20Visualize%20Target%20Variable%20Distribution

### 5. Normalisation
Options:
1. MinMax
2. Z-normalisation
3. Vector Normalisation
4. More research: https://www.geeksforgeeks.org/python/data-normalization-with-pandas/

##### 1. MinMax

In [ ]:
scaler = MinMaxScaler()
dataset[continuous_names] = scaler.fit_transform(dataset[continuous_names])
dataset[continuous_names]

fig,ax=plt.subplots(1,figsize=(20,8))
sns.boxplot(data=dataset[continuous_names],ax=ax)

In [ ]:
# %store dataset